In [ ]:
import io
import os
import glob

folder = '/Users/sebastianx/Corpus Linguistics/BNC/Texts'

try:
    from google.colab import files
    uploaded_files = files.upload()
except ImportError:
    uploaded_files = {}
    for file_path in glob.glob(os.path.join(folder, '*')):
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)
            uploaded_files[filename] = open(file_path, 'rb').read()

print(f"已加载 {len(uploaded_files)} 个文件：")

已加载 4049 个文件：


In [ ]:
import xml.etree.ElementTree as ET
import spacy

# Load the large English model for accurate dependency parsing
nlp = spacy.load('en_core_web_lg')
nlp.max_length = 5000000

# --- Demonstration: xcomp/ccomp = complement of help; acl = modifier on object NP ---
demo_sentences = [
    "the council helped patients wishing to appeal against their detention under the new Act.",
    "Money helps solve problems",
    "Money helps to solve problems",
    "the council helped the patients to appeal against the decision",
]

for sent in demo_sentences:
    doc = nlp(sent)
    help_token = None
    for t in doc:
        if t.lemma_ == 'help' and t.pos_ == 'VERB':
            help_token = t
            break
    print(sent)
    if help_token:
        for child in help_token.children:
            if child.dep_ in {'xcomp', 'ccomp', 'acl'}:
                print(f"  {child.text:10} dep={child.dep_:6} head={child.head.text}")
    print()

In [ ]:
# Shared stop-word sets used by get_dep_var and get_comp_lemma.
# STOP_C5: finite verb and modal tags that signal a new clause boundary.
# STOP_POS: interrogative/relative pronouns and subordinating conjunctions.
# STOP_TEXT: lexical items that introduce non-complement structures:

STOP_C5   = {'VVD', 'VVZ', 'VHD', 'VHZ', 'VBD', 'VBZ', 'VM0',
             'VDD', 'VDZ', 'DOD', 'DOZ'}  # added finite forms of "do" (main verb & auxiliary)
STOP_POS  = {'PNQ', 'CJS', 'CJT'}
STOP_TEXT = {'and', 'or', 'if', 'whether', 'which', 'who', 'whom', 'that',
             'with', 'by', 'for', 'on', 'about', 'instead'}

# BNC tags for "do" as a main verb non-finite forms:
# VDI = infinitive ("to do"), VDB = base form ("help do"), DO0 = auxiliary base form
DO_NONFIN = {'VVI', 'VVB', 'DO0', 'VDI', 'VDB'}


def _spacy_dep_of_verb(sentence_text, help_surface, verb_surface):
    """Use spaCy to find the dependency relation of verb_surface relative to help.

    Returns the dep_ label of the first token matching verb_surface that is a
    child of help (xcomp, ccomp) or a grandchild via an object NP (acl).
    Returns None if the relationship cannot be determined.

    Used to filter out acl post-nominal modifiers misidentified as complements.
    e.g. "helped patients wishing to appeal"  → 'wishing' is acl on 'patients'
         "help the committee set up by X ..."  → 'set' is acl on 'committee'
    """
    doc = nlp(sentence_text)
    help_token = None
    for token in doc:
        if token.text.lower() == help_surface.lower() and token.pos_ == 'VERB':
            help_token = token
            break
    if help_token is None:
        return None

    verb_lower = verb_surface.lower()

    # Direct complement of help: xcomp or ccomp
    for child in help_token.children:
        if child.dep_ in ('xcomp', 'ccomp') and child.text.lower() == verb_lower:
            return child.dep_

    # acl on help's direct object NP
    for child in help_token.children:
        if child.dep_ in ('dobj', 'obj', 'oprd'):
            for grandchild in child.children:
                if grandchild.dep_ == 'acl' and grandchild.text.lower() == verb_lower:
                    return 'acl'

    return None


def get_dep_var(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Determine the type of non-finite complement following 'help' using BNC c5 tags.

    Scans up to 30 words to the right of help, stopping at clause boundaries.
    Punctuation elements (tag='c') do not count toward the word limit.
    When sentence_text and help_surface are supplied, spaCy is used to verify
    that candidate BARE and ING verbs are genuine complements (xcomp/ccomp)
    and not acl post-nominal modifiers on help's object NP.
    Uses c5.startswith() throughout to capture BNC ambiguity tags (e.g. VVG-AJ0).
    Returns one of: 'TO', 'BARE', 'ING', 'INING', or 'NA'.
    """
    if not help_c5 or not help_c5.startswith('VV'):
        return 'NA'
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        # Stop at sentence-ending punctuation
        if text in {'.', '!', '?', ','}:
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        # Stop at finite verbs, modals, or subordinating conjunctions
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO infinitive: "to (adv/neg)* VVI/VVB/DO0/VDI/VDB"
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0'}:
                    continue
                if nxt_c5 in DO_NONFIN:
                    return 'TO'
                break
            break
        # INING: "in (det/pron/noun)* VVG"  — use startswith to catch ambiguity tags e.g. VVG-AJ0
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith('VVG'):
                    return 'INING'
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    continue
                break
        # ING: -ing form — use startswith to catch ambiguity tags e.g. VVG-AJ0
        # verify with spaCy that it is xcomp/ccomp, not acl
        # e.g. "helped patients wishing to appeal" → 'wishing' is acl on 'patients', skip
        if elem.tag == 'w' and c5.startswith('VVG'):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue  # post-nominal participle, not a complement — keep scanning
            return 'ING'
        # BARE: bare infinitive — verify with spaCy that it is xcomp/ccomp, not acl
        if elem.tag == 'w' and c5 in DO_NONFIN:
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return 'BARE'
    return 'NA'


def get_comp_lemma(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return the verb lemma of the non-finite complement clause following 'help'.

    Mirrors the search logic of get_dep_var but returns the BNC headword (hw
    attribute) of the identified verb rather than its construction category.
    Punctuation tokens (tag='c') are skipped and do not count as words.
    acl verbs (ING and BARE) are filtered out via spaCy when sentence_text is supplied.
    Uses c5.startswith() to capture BNC ambiguity tags (e.g. VVG-AJ0).
    Returns 'NA' if no non-finite complement is found within 30 words.
    """
    if not help_c5 or not help_c5.startswith('VV'):
        return 'NA'

    def lemma_of(elem):
        hw = elem.get('hw', '').strip().lower()
        return hw if hw else (elem.text or '').strip().lower()

    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        if text in {'.', '!', '?', ','}:
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO infinitive: return lemma of the infinitive verb
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0'}:
                    continue
                if nxt_c5 in DO_NONFIN:
                    return lemma_of(nxt)
                break
            break
        # INING: return lemma of the -ing verb
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith('VVG'):
                    return lemma_of(nxt)
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    continue
                break
        # ING: use startswith to catch ambiguity tags; verify with spaCy
        if elem.tag == 'w' and c5.startswith('VVG'):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return lemma_of(elem)
        # BARE: verify with spaCy before returning
        if elem.tag == 'w' and c5 in DO_NONFIN:
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return lemma_of(elem)
    return 'NA'


def extract_bnc_sentence(elements, help_idx):
    """Reconstruct the plain text of the BNC sentence containing the help token."""
    sent_start = 0
    for k in range(help_idx, -1, -1):
        if elements[k].tag == 's':
            sent_start = k
            break
    parts = []
    for k in range(sent_start, len(elements)):
        elem = elements[k]
        if k > sent_start and elem.tag == 's':
            break
        if elem.tag in ['w', 'c'] and elem.text:
            parts.append(elem.text)
    return ' '.join(parts)


def get_interv_words(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return the number of words between help and its non-finite complement verb.

    Uses the same BNC c5 tag scanning as get_dep_var.
    intervening_words = (words between help and 'to') + (adverbs/negation between 'to' and verb)
    i.e. equivalent to  i + j - 2  where i = offset to 'to', j = offset from 'to' to verb (1-indexed).

    For BARE/ING/INING: words strictly between help and the complement verb.
    acl post-nominal modifiers (ING and BARE) are skipped via spaCy.
    Uses c5.startswith() to capture BNC ambiguity tags (e.g. VVG-AJ0).
    Returns an integer, or 'NA' if no complement is found.
    """
    if not help_c5 or not help_c5.startswith('VV'):
        return 'NA'
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        if text in {'.', '!', '?', ','}:
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO: words_before_to + adverbs/negation between 'to' and verb
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            words_before_to = word_count - 1
            adv_count = 0  # adverbs/negations (AV0/XX0) between 'to' and the complement verb
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0'}:
                    adv_count += 1
                    continue
                if nxt_c5 in DO_NONFIN:
                    return words_before_to + adv_count
                break
            break
        # INING: words before 'in' + NP words between 'in' and gerund
        if elem.tag == 'w' and text == 'in':
            words_before_in = word_count - 1
            np_count = 0  # determiner/pronoun/noun words between 'in' and the gerund
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith('VVG'):
                    return words_before_in + np_count
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    np_count += 1
                    continue
                break
        # ING: use startswith; verify with spaCy, skip if acl
        if elem.tag == 'w' and c5.startswith('VVG'):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return word_count - 1
        # BARE: verify with spaCy, skip if acl
        if elem.tag == 'w' and c5 in DO_NONFIN:
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return word_count - 1
    return 'NA'

In [ ]:
# Create KWIC concordance lines for "help" from our XML files

# Output lists — ordered to match the final spreadsheet column layout:
# Hit | KWIC | Form | POS | DepVar | CompLemma | IntervWords |
# TextID | Year | Genre | Subgenre | Mode | Variety | Corpus |
# SpeakerID | SpeakerGender | SpeakerAge | SpeakerSoc
hits         = []
help_KWIC    = []
help_form    = []
help_pos     = []
dep_var      = []
comp_lemma   = []
interv_words = []
file_id      = []
year         = []
genre        = []
subgenre     = []
mode         = []
variety      = []
corpus       = []
speaker_id     = []
speaker_gender = []
speaker_age    = []
soc            = []

hit = 1
max_buffer = 30  # for left context, keep last 30 words

# BNC age group codes → midpoint of age range
age_midpoint = {
    'Ag0': 'unknown',
    'Ag1': 7,    # 0-14
    'Ag2': 20,   # 15-24
    'Ag3': 30,   # 25-34
    'Ag4': 40,   # 35-44
    'Ag5': 55,   # 45+
    'X':   'unknown',
}

XML_NS = '{http://www.w3.org/XML/1998/namespace}'


for file, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    # --- (1) File-level metadata ---

    id = file.replace('.xml', '')

    classcode = tree.find('.//classCode')
    meta = classcode.text

    creation = tree.find('.//creation')
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'

    if meta[0] == "W":
        modevalue     = "Written"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue    = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    elif meta[0] == "S":
        modevalue     = "Spoken"
        genrevalue    = "NA"
        subgenrevalue = "NA"
        person = tree.find('.//person')
        if person is not None:
            spk_id     = person.get(f'{XML_NS}id', 'NA')
            spk_gender = person.get('sex', 'NA')
            raw_age    = person.get('ageGroup', 'Ag0')
            spk_age    = age_midpoint.get(raw_age, 'unknown')
            spk_soc    = person.get('soc', 'NA')
        else:
            spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "unknown", "NA"

    else:
        modevalue     = "Unknown"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue    = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    # --- (2) KWIC concordance & annotation ---

    left_context = []
    collecting_right = False
    right_context = []
    current_right_context_words = 0

    all_elems = list(tree.iter())

    for elem_idx, elem in enumerate(all_elems):

        # Collect right context for the previous hit
        if collecting_right:
            if elem.tag in ['w', 'c']:
                right_context.append(elem.text)
                if elem.tag == 'w':
                    current_right_context_words += 1
                    if current_right_context_words >= 60:
                        collecting_right = False
                        right_KWIC = ' '.join(filter(None, right_context))
                        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

        # Build rolling left context buffer (30 words)
        if elem.tag in ['w', 'c'] and elem.text and not elem.text.lower().startswith('help'):
            left_context.append(elem.text)
            if len(left_context) > max_buffer:
                left_context.pop(0)

        # Found a "help" hit
        if elem.tag == 'w' and elem.text and elem.text.lower().startswith('help'):

            # Store right context of any preceding hit still being collected
            if collecting_right:
                right_KWIC = ' '.join(filter(None, right_context))
                help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

            left_KWIC = ' '.join(left_context)
            help_word = elem.text.strip()   # strip whitespace for spaCy matching
            help_c5   = elem.get('c5')

            # Sentence text for spaCy (acl filtering in get_dep_var / get_comp_lemma)
            sentence_text = extract_bnc_sentence(all_elems, elem_idx)

            # Append all values in column order
            hits.append(str(hit))
            # help_KWIC appended once right context is complete (above / end of file)
            help_form.append(help_word)
            help_pos.append(help_c5)
            dep_var.append(get_dep_var(all_elems, elem_idx, help_c5,
                                       sentence_text, help_word))
            comp_lemma.append(get_comp_lemma(all_elems, elem_idx, help_c5,
                                             sentence_text, help_word))
            interv_words.append(get_interv_words(all_elems, elem_idx, help_c5,
                                                  sentence_text, help_word))
            file_id.append(id)
            year.append(yearvalue)
            genre.append(genrevalue)
            subgenre.append(subgenrevalue)
            mode.append(modevalue)
            variety.append("BrE")
            corpus.append("BNC")
            speaker_id.append(spk_id)
            speaker_gender.append(spk_gender)
            speaker_age.append(spk_age)
            soc.append(spk_soc)

            hit += 1
            collecting_right = True
            right_context = []
            current_right_context_words = 0

    # Store right context of the last hit in this file
    if collecting_right:
        right_KWIC = ' '.join(filter(None, right_context))
        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

# --- (3) Build DataFrame and export ---

import pandas as pd

df_all = pd.DataFrame({
    'Hit':           hits,
    'KWIC':          help_KWIC,
    'Form':          help_form,
    'POS':           help_pos,
    'DepVar':        dep_var,
    'CompLemma':     comp_lemma,
    'IntervWords':   interv_words,
    'TextID':        file_id,
    'Year':          year,
    'Genre':         genre,
    'Subgenre':      subgenre,
    'Mode':          mode,
    'Variety':       variety,
    'Corpus':        corpus,
    'SpeakerID':     speaker_id,
    'SpeakerGender': speaker_gender,
    'SpeakerAge':    speaker_age,
    'SpeakerSoc':    soc,
})

df_written = df_all[df_all['Mode'] == 'Written'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar', 'CompLemma', 'IntervWords',
    'TextID', 'Year', 'Genre', 'Subgenre', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_spoken = df_all[df_all['Mode'] == 'Spoken'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar', 'CompLemma', 'IntervWords',
    'TextID', 'SpeakerID', 'Year', 'SpeakerGender', 'SpeakerAge', 'SpeakerSoc',
    'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_written.to_excel('BNC_help_written.xlsx', index=False)
df_spoken.to_excel('BNC_help_spoken.xlsx', index=False)

print(f"Written: {len(df_written)} results → BNC_help_written.xlsx")
print(f"Spoken:  {len(df_spoken)} results → BNC_help_spoken.xlsx")